In [2]:
import torch
import torch.nn as nn

# Regularization: L1 and L2 
 
## What is regularization?
 
Regularization is a technique to **reduce overfitting** by adding a penalty to the loss function. The model is forced to balance two objectives:
 
$$\mathcal{L}_{reg} = \underbrace{\mathcal{L}_{original}}_{\text{training error}} + \underbrace{\lambda \cdot \Omega(w)}_{\text{complexity penalty}}$$
 
The penalty always adds a positive value to the loss — yes, the loss increases. This is intentional: the model forgoes making fewer mistakes in training to generalize better on the test.

## L1 — Lasso
 
### Formula
 
$$\mathcal{L} = \mathcal{L}_{original} + \lambda \sum_{i=1}^{n} |w_i|$$
 
Takes all weights, applies **absolute value** and adds them together.
 
**Example with 4 weights:**
"'
w = [3, -2, 0.5, -1]
|3| + |-2| + |0.5| + |-1| = 3 + 2 + 0.5 + 1 = 6.5
 
Loss = Loss_original + λ * 6.5
"'
 
### Gradient
 
$$\frac{\partial \mathcal{L}}{\partial w_i} = \frac{\partial \mathcal{L}_{original}}{\partial w_i} + \lambda \cdot \text{sign}(w_i)$$
 
### Weight Update
 
$$w_i \leftarrow w_i - \eta \left( \frac{\partial \mathcal{L}_{original}}{\partial w_i} + \lambda \cdot \text{sign}(w_i) \right)$$
 
### Why does L1 reset weights?
 
The gradient of the penalty is 'sign(w)' — **constant, regardless of weight size**:
 
"'
w = 10 → push = 1
w = 1 → push = 1
w = 0.1 → push = 1
w = 0.01 → push = 1 ← does not decrease!
"'
 
The weight continues to be pushed with the same force until it crosses zero — hence **zero**.
 
### When to use
 
- Many features, few really relevant
- Want **automatic feature selection**
- Want a sparse and interpretable model


In [ ]:
l1_lambda = 1e-5
l1_penalty = sum(p.abs().sum() for p in model.parameters())

loss += (l1_lambda * l1_penalty)


## L2 — Ridge
 
### Formula
 
$$\mathcal{L} = \mathcal{L}_{original} + \lambda \sum_{i=1}^{n} w_i^2$$
 
Takes all the weights, raises to **square** and adds them together.
 
**Example with 4 weights:**
"'
w = [3, -2, 0.5, -1]
3² + (-2)² + 0.5² + (-1)² = 9 + 4 + 0.25 + 1 = 14.25
 
Loss = Loss_original + λ * 14.25
"'
 
### Gradient
 
$$\frac{\partial \mathcal{L}}{\partial w_i} = \frac{\partial \mathcal{L}_{original}}{\partial w_i} + 2\lambda w_i$$
 
### Weight Update
 
$$w_i \leftarrow w_i - \eta \left( \frac{\partial \mathcal{L}_{original}}{\partial w_i} + 2\lambda w_i \right)$$
 
Reorganizing:
 
$$w_i \leftarrow \underbrace{w_i(1 - 2\eta\lambda)}_{\text{weight decay}} - \eta \cdot \frac{\partial \mathcal{L}_{original}}{\partial w_i}$$
 
The term $(1 - 2\eta\lambda)$ **decays the weight at each step** — hence L2 is also called *weight decay*.
 
### Why doesn't L2 reset weights?
 
The gradient is proportional to the weight — **the lower the weight, the less push**:
 
"'
w = 10 → push = 20
w = 1 → push = 2
w = 0.1 → push = 0.2
w = 0.01 → push = 0.02
w → 0 → push → 0 ← disappear before zeroing
"'
 
### When to use
 
- All features likely contribute
- There are features correlated with each other
- Want numerical stability
- **Default when you don't know which one to use**


In [ ]:
optim = torch.optim.Adam(
    weight_decay=1e-4  # Weight decay it's the regularization (lambda value)
)

## Adam's problem with weight decay
 
### Adam normalizes the gradient
 
Adam doesn't use pure gradient — it normalizes by history:
 
$$w \leftarrow w - \eta \cdot \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon}$$
 
Where 'm̂' is the mean and 'v̂' is the variance of the past gradients.
 
### L2 in Adam is distorted
 
When you pass 'weight_decay' in 'torch.optim.Adam', PyTorch adds L2 **in the gradient**, before normalization:
 
```python
# Internally Adam does:
grad = grad + weight_decay * w # ← L2 goes in here
w = w - Adam(grad) # ← normalize everything together
```
 
The L2 penalty is scaled along with the gradient — the actual effect of decay is distorted and unpredictable.
 
```
Adam + L2 in loss:
[gradient + λw] → Adam normalization → update
        ↑
   distorted penalty here
```
 
### AdamW solves the problem
 
AdamW **separates** the weight decay from the gradient, applying it directly to the weight after normalization:
 
```python
# Internally AdamW does:
w = w - Adam(grad) # ← normalize only the gradient
w = w - weight_decay * w # ← decay applied later, clean
```

 
| | SGD | Adam | AdamW |
|---|---|---|---|
| L2 == weight decay? | Yes | No | Yes |
| Recommended | weight_decay | **AdamW** | **AdamW** |
 
> **Rule of thumb:** always use **AdamW** instead of Adam when you want L2 regularization. It is the standard in modern deep learning.
